<a href="https://colab.research.google.com/github/Kareena-3/FlyRank-AI/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — Research Question & Provisional Lane

## Provisional lane: CTR / Engagement Opportunity Scoring

I am choosing the **CTR / Engagement Opportunity Scoring** lane as my provisional project direction for the next seven weeks. The lane guide defines this as finding visible pages that under-capture clicks or engagement and deserve review, using position/volume-adjusted analysis. My focus will initially be on CTR, with engagement signals considered later if the data supports them.

## Research question

**Which visible pages have lower-than-expected CTR for their search position, and which of those pages should be prioritized for review?**

I want to compare a page with other pages at a similar position rather than treating a low raw CTR as automatically problematic. The eventual analysis will use minimum-volume thresholds and position-adjusted expected CTR so that low-volume noise and the natural CTR difference between positions do not dominate the recommendations.

## Unit of analysis

**One page/content item.** Each row in the starter dataset represents a content item/page for the analysis. For the larger warehouse work, I will explicitly define the page/content grain and the relevant observation window before modeling.

## Intended output

A **ranked CTR opportunity list** showing which pages appear most worth reviewing, including the position-adjusted CTR gap and short reason codes such as high impressions, strong position, low CTR, and sufficient volume.

## Action someone could take

A content/SEO practitioner could use the ranked list to decide which pages to review first for **title/meta improvements, intent or snippet alignment, on-page engagement improvements, or monitoring**. A low score would not automatically mean that a page needs rewriting; it would identify a page for human review.

## Cost of a wrong recommendation

- **False positive:** time is spent reviewing or changing a page that did not actually have a meaningful CTR opportunity.
- **False negative:** a genuinely valuable CTR opportunity is missed, potentially leaving clicks on the table.
- **Important risk:** a low CTR can have legitimate causes, so the system must not turn an observational signal into an automatic recommendation to rewrite content.

## Why data / ML can help

There may be many pages to review, and CTR depends strongly on search position, impressions, content type, intent, age/freshness, and other observable signals. Data can quantify these relationships and provide a consistent ranking instead of relying only on manual inspection. ML may eventually help combine several signals and rank opportunities, but **the project is not simply 'train a model'**: the core work is defining the decision, controlling for position and volume, building a transparent baseline, validating it, and turning the evidence into safe ranked actions.

## What I will treat as success

The project should produce a ranked list that is useful for human review, with clear reasons for each recommendation and an honest validation of how reliable the ranking is. I will describe findings as **observed associations**, not as proof of Google's ranking algorithm or proof that changing a page will cause a particular outcome.

## Evidence from the starter dataset

The starter dataset contains **30,000 pages**. The following code loads the supplied CSV and checks the position/content-type CTR pattern that motivated this lane.

In [2]:
import os
import pandas as pd
from pathlib import Path

# Your GitHub repository
REPO_URL = "https://github.com/Kareena-3/FlyRank-AI.git"
REPO_DIR = Path("/content/FlyRank-AI")

# Clone your repo into the Colab runtime if it isn't already there
if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

# Move into the root of YOUR repository
os.chdir(REPO_DIR)

# Dataset location inside your repo
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

print("Working directory:", Path.cwd())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH.resolve())

# Load the starter dataset
df = pd.read_csv(DATA_PATH)

print(f"\nRows: {len(df)}")

visible = df[df["impressions_90d"] >= 100].copy()

print(
    f"Pages with impressions_90d >= 100: {len(visible)}"
)

ctr_by_content_pos = (
    visible
    .groupby(["position_tier", "content_type"])["ctr"]
    .mean()
    .reset_index()
)

selected = ctr_by_content_pos[
    ctr_by_content_pos["position_tier"].isin(["page_1", "striking"])
].sort_values(["position_tier", "ctr"])

print(
    "\nMean CTR by position tier and content type (selected results):"
)

print(
    selected.to_string(
        index=False,
        formatters={"ctr": "{:.4f}".format}
    )
)

Cloning into '/content/FlyRank-AI'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 91 (delta 12), reused 63 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.84 MiB | 9.15 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Working directory: /content/FlyRank-AI
Dataset exists: True
Dataset path: /content/FlyRank-AI/data/raw/content_refresh_anonymized.csv

Rows: 30000
Pages with impressions_90d >= 100: 22006

Mean CTR by position tier and content type (selected results):
position_tier       content_type    ctr
       page_1 comparison article 0.1412
       page_1    keyword article 0.3458
       page_1     feedly article 0.9048
     striking comparison article 0.1474
     striking    keyword article 0.2559
     striking     feedly article 0.3580


### What these numbers suggest

At the **same position tier**, the content types show materially different observed CTRs. For example, on page 1, comparison articles have mean CTR **0.1412**, compared with **0.3458** for keyword articles. In the striking tier, comparison articles are at **0.1474**, versus **0.2559** for keyword articles.

This does **not** prove that content type causes lower CTR. It does show that a position-aware CTR opportunity analysis is worth investigating: a page can have a CTR that looks low for reasons that are easier to see when position and other context are considered.

## Provisional plan for the next 7 weeks

1. Define a leakage-safe data contract and page-level analysis grain.
2. Audit CTR, impressions, position, content type, intent, freshness, and engagement signals.
3. Build a simple position-adjusted CTR baseline and minimum-volume filter.
4. Compare that baseline with a more flexible model only if the added complexity improves the ranking.
5. Validate the ranking honestly and inspect the top candidates manually.
6. Produce reason codes and action suggestions rather than opaque scores.

**Status:** This is a provisional lane and can be changed later if the warehouse evidence suggests a stronger question.